In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

MY_IP = "192.168.0.74"
MAX_LEN = 2000

In [2]:
#data path
np.load(r"C:\Users\User\Downloads\Nika\raw\raw\tor_200w_2500tr.npz", allow_pickle=True)
print(dataF.files)

['data', 'labels']


In [3]:
print("dataF keys:", dataF.files)

dataF keys: ['data', 'labels']


In [4]:
# Load arrays
X_F = dataF['data']
y_F = dataF['labels']

print("Total X shape:", X_F)
print("Total y shape:", y_F)

Total X shape: [[ 1  1 -1 ... -1 -1 -1]
 [ 1  1 -1 ... -1 -1 -1]
 [ 1  1 -1 ... -1 -1 -1]
 ...
 [ 1  1 -1 ... -1 -1 -1]
 [ 1  1 -1 ... -1 -1 -1]
 [ 1  1  1 ... -1 -1 -1]]
Total y shape: ['9gag.com' '9gag.com' '9gag.com' ... 'upornia.com' 'upornia.com'
 'upornia.com']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_F,
    y_F,
    test_size=0.2,
    random_state=42,
    stratify=y_F
)
X_train_small=X_train.astype(np.float32)
X_test_small=X_test.astype(np.float32)
print("Train:", X_train_small.shape)
print("Test:", X_test_small.shape)

Train: (400000, 5000)
Test: (100000, 5000)


In [ ]:
# Encode labels
le = LabelEncoder()

y_encoded = le.fit_transform(y_all)

num_classes = len(np.unique(y_encoded))

y_cnn = to_categorical(
    y_encoded,
    num_classes=num_classes
)

# Split again
X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_cnn,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# Reshape for CNN
X_train = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)

In [ ]:
model = Sequential()

model.add(
    Conv1D(
        filters=32,
        kernel_size=3,
        activation='relu',
        input_shape=(X_train.shape[1], 1)
    )
)

model.add(
    MaxPooling1D(pool_size=2)
)

model.add(Flatten())

model.add(
    Dense(64, activation='relu')
)

model.add(
    Dense(num_classes, activation='softmax')
)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

In [ ]:
loss, acc_cnn = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("CNN Accuracy:", acc_cnn)

In [ ]:
y_pred_cnn = model.predict(X_test)

y_pred_labels = np.argmax(y_pred_cnn, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

precision = precision_score(
    y_true_labels,
    y_pred_labels,
    average='macro'
)

recall = recall_score(
    y_true_labels,
    y_pred_labels,
    average='macro'
)

print("CNN Precision:", precision)
print("CNN Recall:", recall)